# 2. Házi feladat — Differenciálható volumetrikus renderelés

Ebben a házi feladatban egy egyszerű **NeRF-szerű** rendszert fogsz építeni
PyTorch-ban. A feladat lényege az **inverz renderelés**:

$$
\text{jelenet-paraméterek} \;\longrightarrow\; \text{renderelt kép}
\;\longrightarrow\; \text{képi veszteség}
\;\longrightarrow\; \text{gradiens a jelenet-paraméterekre}.
$$

Vagyis nem közvetlenül a 3D mezőt tanítjuk felügyelt célértékekkel, hanem
képeket renderelünk belőle, ezeket hasonlítjuk össze célképekkel, és a
gradiensek a renderelőn keresztül jutnak vissza a tanulható 3D reprezentációba.

A notebook a korábbi PyTorch-anyaghoz hasonló tanítási mintát használ:

```python
pred = model(...)
loss = mse_criterion(pred, target)
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

A különbség az, hogy itt a `pred` nem közvetlen osztályozási vagy denoising
kimenet, hanem egy **differenciálható renderer** által előállított kép.

## Kötelező és opcionális részek

| Rész | Kötelező? | Mit kell implementálni? |
|---|---:|---|
| 1. Volumetrikus renderelés egy sugárra | igen | `compute_alphas`, `compute_transmittance`, `volume_render_ray` |
| 2. Voxel-rácsos inverz renderelés | igen | `voxel_query` |
| 3. Tiny-NeRF | igen | `PositionalEncoding.forward`, `TinyNeRF.forward` |
| 4. Gauss-primitívmező, 3DGS-inspirált | nem | opcionális futtatás és összehasonlítás |

**A feladat célja:** a fő matematikai és PyTorch-lépések világos
implementálása. Nem kell saját renderelő-könyvtárat, CUDA kódot vagy bonyolult
3D grafikai infrastruktúrát írnod.

> **Tipp:** futtasd a notebookot Google Colab-ban GPU-val. A notebook
> alapértelmezésben `FAST_MODE=True`, ami elég a beadáshoz és a hibakereséshez.
> Ha szebb képeket szeretnél, állítsd `FAST_MODE=False`-ra.


In [ ]:
# Környezet beállítása
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'torch', 'torchvision', 'matplotlib', 'numpy', 'tqdm', '--quiet'
    ])

import math
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Eszköz: {device}')

# A gyors mód célja, hogy a notebook gyorsan kipróbálható legyen.
# GPU-n már nézhető képeket ad; CPU-n szándékosan nagyon kicsi beállításokat
# használ, hogy a notebook ne tűnjön „lefagyottnak”.
FAST_MODE = True

if FAST_MODE and device.type == 'cpu':
    IMG_SIZE = 16
    GRID_RES = 20
    N_TRAIN = 4
    N_TEST = 2

    RAY_BATCH = 64
    N_SAMPLES_TRAIN = 8
    N_SAMPLES_EVAL = 10
    N_ITERS_VOXEL = 100
    N_ITERS_NERF = 150
    RENDER_CHUNK = 1024
    NERF_HIDDEN = 32
    NERF_LAYERS = 2

    N_GAUSS_SPLAT = 64
    RAY_BATCH_SPLAT = 32
    N_SAMPLES_SPLAT = 8
    N_SAMPLES_SPLAT_EVAL = 10
    N_ITERS_SPLAT = 80
    RENDER_CHUNK_SPLAT = 64
elif FAST_MODE:
    IMG_SIZE = 32
    GRID_RES = 32
    N_TRAIN = 8
    N_TEST = 4

    RAY_BATCH = 384
    N_SAMPLES_TRAIN = 24
    N_SAMPLES_EVAL = 32
    N_ITERS_VOXEL = 400
    N_ITERS_NERF = 800
    RENDER_CHUNK = 2048
    NERF_HIDDEN = 128
    NERF_LAYERS = 4

    # Az opcionális Gauss-rész külön, kisebb beállításokat kap,
    # mert a pontok × Gaussok páronkénti kiértékelése memóriaigényes.
    N_GAUSS_SPLAT = 192
    RAY_BATCH_SPLAT = 96
    N_SAMPLES_SPLAT = 24
    N_SAMPLES_SPLAT_EVAL = 32
    N_ITERS_SPLAT = 300
    RENDER_CHUNK_SPLAT = 192
else:
    IMG_SIZE = 64
    GRID_RES = 64
    N_TRAIN = 16
    N_TEST = 4

    RAY_BATCH = 1024
    N_SAMPLES_TRAIN = 64
    N_SAMPLES_EVAL = 96
    N_ITERS_VOXEL = 1500
    N_ITERS_NERF = 3000
    RENDER_CHUNK = 4096
    NERF_HIDDEN = 128
    NERF_LAYERS = 4

    N_GAUSS_SPLAT = 1024
    RAY_BATCH_SPLAT = 1024
    N_SAMPLES_SPLAT = 64
    N_SAMPLES_SPLAT_EVAL = 64
    N_ITERS_SPLAT = 1500
    RENDER_CHUNK_SPLAT = 512

mse_criterion = nn.MSELoss()

print('FAST_MODE =', FAST_MODE)
print(f'Kép: {IMG_SIZE}×{IMG_SIZE}, rács: {GRID_RES}³, '
      f'train nézetek: {N_TRAIN}, teszt nézetek: {N_TEST}')


---

## 1. Rész — Volumetrikus renderelés egyetlen sugárra

A volumetrikus renderelési integrál egy sugár mentén:

$$
C(\mathbf{r}) =
\int_{t_n}^{t_f}
T(t)\,\sigma(\mathbf{r}(t))\,\mathbf{c}(\mathbf{r}(t))\,\mathrm{d}t,
\qquad
T(t) =
\exp\!\left(
-\int_{t_n}^{t}\sigma(\mathbf{r}(s))\,\mathrm{d}s
\right).
$$

Itt $\sigma$ a sűrűség (*density*), $\mathbf{c}$ a szín, $T$ pedig a
transzmittancia: annak mértéke, hogy a sugár mentén mennyi fény jut el az
adott pontig.

A sugár mentén $N$ mintavételi pontot véve, $t_1<\dots<t_N$ helyeken, a
diszkrét közelítés klasszikus **front-to-back alpha-kompozíció** alakot ölt:

$$
\alpha_i = 1 - \exp(-\sigma_i\,\delta_i),
\qquad
T_i = \prod_{j<i}(1-\alpha_j),
\qquad
w_i = T_i\,\alpha_i,
\qquad
\hat{C} = \sum_{i=1}^{N} w_i\,\mathbf{c}_i.
$$

A kódban minden mintához tartozik egy `delta_i`, ezért az utolsó mintához
egyszerűen megismételjük az előző lépésközt. Ez csak egy praktikus
diszkretizációs konvenció, hogy a `sigma`, `color` és `delta` tenzoroknak
ugyanaz legyen a mintaszám-dimenziója.

Az alábbi három függvényt fogod implementálni — később ezek lesznek a
3D renderelő építőkövei is.


In [ ]:
def compute_alphas(sigma: torch.Tensor, delta: torch.Tensor) -> torch.Tensor:
    """alpha_i = 1 - exp(-sigma_i * delta_i)."""
    # TODO: Implementáld a fenti képletet.
    raise NotImplementedError("Implementáld a compute_alphas függvényt!")


In [ ]:
# ── Ellenőrzés ──
_sigma = torch.tensor([0.0, 1.0, 10.0])
_delta = torch.tensor([0.1, 0.1, 0.1])
_alpha = compute_alphas(_sigma, _delta)

expected = torch.tensor([0.0, 1.0 - math.exp(-0.1), 1.0 - math.exp(-1.0)])
print('alpha =', _alpha)
assert torch.allclose(_alpha, expected, atol=1e-6), 'HIBA a compute_alphas függvényben!'
print('alpha ellenőrzés – OK')


In [ ]:
def compute_transmittance(alpha: torch.Tensor) -> torch.Tensor:
    """T_i = prod_{j<i} (1 - alpha_j); T_0 = 1."""
    # TODO: Készíts (..., N) tenzort, ahol T_0 = 1,
    # T_i = prod_{j<i} (1 - alpha_j).
    # Tipp: kezdő 1-es, konkatenálás, majd torch.cumprod az utolsó dimenzión.
    raise NotImplementedError("Implementáld a compute_transmittance függvényt!")


In [ ]:
# ── Ellenőrzés ──
_alpha = torch.tensor([0.5, 0.5, 0.5])
_T = compute_transmittance(_alpha)
print('T =', _T)
assert torch.allclose(_T, torch.tensor([1.0, 0.5, 0.25]), atol=1e-5),     'HIBA a compute_transmittance függvényben!'
print('T ellenőrzés – OK')


In [ ]:
def volume_render_ray(sigma: torch.Tensor,
                      color: torch.Tensor,
                      delta: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Diszkrét volumetrikus renderelés egy (vagy több) sugárra.

    Args:
        sigma: (..., N) sűrűségek.
        color: (..., N, 3) színek RGB ∈ [0,1].
        delta: (..., N) lépésközök.
    Returns:
        rgb:     (..., 3) integrált szín.
        weights: (..., N) súlyok, w_i = T_i * alpha_i.
    """
    # TODO:
    #   1. alpha = compute_alphas(...)
    #   2. T = compute_transmittance(...)
    #   3. weights = T * alpha
    #   4. rgb = súlyozott színösszeg a minták dimenziója mentén
    # Figyelj: a weights alakja (..., N), ezért szorzás előtt kell egy extra dimenzió.
    raise NotImplementedError("Implementáld a volume_render_ray függvényt!")


In [ ]:
# ── Ellenőrzés: volume_render_ray ──

# 1) Zérus sűrűség: semmi nem látszik, minden súly nulla.
_sigma = torch.zeros(2, 4)
_color = torch.rand(2, 4, 3)
_delta = torch.ones(2, 4) * 0.1
_rgb, _weights = volume_render_ray(_sigma, _color, _delta)
assert _rgb.shape == (2, 3)
assert _weights.shape == (2, 4)
assert torch.allclose(_rgb, torch.zeros(2, 3), atol=1e-6)
assert torch.allclose(_weights, torch.zeros(2, 4), atol=1e-6)

# 2) Nagyon sűrű első minta: az első szín dominál.
_sigma = torch.tensor([[1000.0, 0.0, 0.0]])
_delta = torch.tensor([[1.0, 1.0, 1.0]])
_color = torch.tensor([[[1.0, 0.0, 0.0],
                        [0.0, 1.0, 0.0],
                        [0.0, 0.0, 1.0]]])
_rgb, _weights = volume_render_ray(_sigma, _color, _delta)
assert _rgb[0, 0] > 0.99, 'Az első, majdnem opak minta színének kell dominálnia.'

print('volume_render_ray ellenőrzés – OK')


In [ ]:
# ── Vizualizáció: szintetikus 1D sugár ──
# Egy "köd" + "fal" jelenet: alacsony szórt sűrűség + egy kemény réteg t≈0.7-nél.
N = 128
t = torch.linspace(0.0, 1.0, N)
delta = torch.full_like(t, 1.0 / (N - 1))

# Sűrűség: egy Gauss-csúcs (a "fal") + halvány konstans háttér ("köd").
sigma = 0.3 + 60.0 * torch.exp(-((t - 0.7) ** 2) / 0.0025)

# Szín a sugár mentén: balról kék köd, a fal vörös.
color = torch.zeros(N, 3)
color[..., 2] = 0.6                      # kékes köd
wall_mask = (t > 0.65) & (t < 0.75)
color[wall_mask] = torch.tensor([0.9, 0.1, 0.1])  # vörös fal

rgb, weights = volume_render_ray(sigma, color, delta)
print(f'Renderelt RGB = {rgb.numpy().round(3)}')
print(f'Súlyok összege = {weights.sum().item():.3f}  (≈ 1, ha a sugár "sűrűbe" ér)')

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].plot(t, sigma); axes[0].set_title('σ(t)'); axes[0].set_xlabel('t')
axes[1].plot(t, weights); axes[1].set_title('w(t) = T(t)·α(t)'); axes[1].set_xlabel('t')
axes[2].imshow(rgb[None, None, :].numpy(), extent=[0, 1, 0, 0.2])
axes[2].set_title('Integrált szín')
axes[2].set_yticks([])
plt.tight_layout(); plt.show()


---

## 2. Rész — Procedurális hangulatjel illesztése voxel-ráccsal

Ebben a részben egy 3D **voxel-rácsot** illesztünk multi-view képekre.
A "jelenetet" egy zárt képletű analitikus mező adja: egy sárga fej, két fekete
szem és hét kicsi piros gömb a mosolyhoz. A bemeneti képeket ebből az
analitikus mezőből rendereljük, majd a voxel-rács feladata ezeknek a képeknek
a reprodukálása.

Ez az inverz renderelés első konkrét példája a notebookban:

```text
voxel-rács paraméterei -> render_rays(...) -> RGB kép -> MSE veszteség
```

A tanítás során **nem** adunk közvetlen célértéket a voxelrács egyes celláira.
Csak azt mondjuk meg, hogy a belőle renderelt képek hasonlítsanak a célképekre.

A jelenetet a `smiley_field` adja: bármely 3D pontban visszaadja a
$(\sigma, \mathbf{c})$ párt. Ezt a függvényt megkapod — Te az ehhez illeszkedő
voxel-rácsot fogod megtanulni.


In [ ]:
# ── Procedurális jelenet (adott) ──
# A smiley-t kis, sima szélű gömbök összegeként definiáljuk.
# A képlet vektorizált: egyszerre számolja ki az összes primitív hozzájárulását,
# ezért CPU-n is lényegesen gyorsabb, mint egy Python-ciklusos változat.
_HEAD_C  = torch.tensor([1.00, 0.85, 0.10])   # sárga
_EYE_C   = torch.tensor([0.05, 0.05, 0.05])   # majdnem fekete
_MOUTH_C = torch.tensor([0.85, 0.10, 0.10])   # piros

_centers = [[0.0, 0.0, 0.0]]
_radii = [0.55]
_sharpness = [100.0]
_density_scale = [20.0]
_colors = [_HEAD_C]

# Bal és jobb szem.
for cx in (-0.20, +0.20):
    _centers.append([cx, 0.18, 0.42])
    _radii.append(0.085)
    _sharpness.append(220.0)
    _density_scale.append(50.0)
    _colors.append(_EYE_C)

# Mosoly: hét kis piros gömb egy íven.
for ang in torch.linspace(-1.0, 1.0, 7).tolist():
    x = 0.30 * math.sin(ang)
    y = -0.20 - 0.07 * math.cos(ang)
    z = 0.43
    _centers.append([x, y, z])
    _radii.append(0.055)
    _sharpness.append(220.0)
    _density_scale.append(50.0)
    _colors.append(_MOUTH_C)

_SMILEY_CENTERS = torch.tensor(_centers, dtype=torch.float32)
_SMILEY_RADII = torch.tensor(_radii, dtype=torch.float32)
_SMILEY_SHARPNESS = torch.tensor(_sharpness, dtype=torch.float32)
_SMILEY_DENSITY_SCALE = torch.tensor(_density_scale, dtype=torch.float32)
_SMILEY_COLORS = torch.stack(_colors, dim=0).float()


def smiley_field(p: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """A jelenet (sigma, color) értéke a 3D pontokban.

    Args:
        p: (..., 3) világkoordináták.
    Returns:
        sigma: (...,) sűrűség.
        color: (..., 3) szín.
    """
    p = p.float()
    dev = p.device
    centers = _SMILEY_CENTERS.to(dev)
    radii = _SMILEY_RADII.to(dev)
    sharpness = _SMILEY_SHARPNESS.to(dev)
    density_scale = _SMILEY_DENSITY_SCALE.to(dev)
    colors = _SMILEY_COLORS.to(dev)

    # d alakja: (..., M), ahol M a primitívek száma.
    d = (p[..., None, :] - centers).norm(dim=-1)
    part_sigma = torch.sigmoid(sharpness * (radii - d)) * density_scale

    sigma_total = part_sigma.sum(dim=-1)
    rgb_num = (part_sigma[..., None] * colors).sum(dim=-2)
    color = rgb_num / (sigma_total[..., None] + 1e-8)
    return sigma_total, color


# ── Gyors ellenőrzés néhány pontra ──
_p = torch.tensor([[0.0, 0.0, 0.0],      # fej középpontja
                   [-0.20, 0.18, 0.42], # bal szem
                   [0.0, -0.27, 0.43],  # mosoly közepe
                   [2.0, 2.0, 2.0]])    # üres tér
_s, _c = smiley_field(_p)
for pt, s, c in zip(_p, _s, _c):
    print(f'p={pt.tolist()}  σ={s.item():6.2f}  rgb={c.numpy().round(2)}')


In [ ]:
# ── Kamerák (adott) ──
def look_at(eye: torch.Tensor,
            target: torch.Tensor | None = None,
            up: torch.Tensor | None = None) -> torch.Tensor:
    """Jobbsodrású look-at; visszaadja a 4x4 camera→world mátrixot.

    A kamera saját rendszerében -z előre, +x jobbra, +y felfelé.
    """
    if target is None:
        target = torch.zeros(3, dtype=eye.dtype, device=eye.device)
    if up is None:
        up = torch.tensor([0.0, 1.0, 0.0], dtype=eye.dtype, device=eye.device)
    forward = (target - eye); forward = forward / forward.norm()
    right = torch.cross(forward, up, dim=-1); right = right / right.norm()
    cam_up = torch.cross(right, forward, dim=-1)
    R = torch.stack([right, cam_up, -forward], dim=1)
    c2w = torch.eye(4, dtype=eye.dtype, device=eye.device)
    c2w[:3, :3] = R
    c2w[:3, 3] = eye
    return c2w


def make_camera_ring(n: int, radius: float = 2.2,
                     elev_deg: float = 25.0,
                     azim_offset_deg: float = 0.0) -> torch.Tensor:
    """n kamera egyenletes elosztással egy szélességi körön."""
    poses = []
    for i in range(n):
        az = math.radians(azim_offset_deg + 360.0 * i / n)
        el = math.radians(elev_deg)
        eye = radius * torch.tensor([
            math.cos(el) * math.sin(az),
            math.sin(el),
            math.cos(el) * math.cos(az),
        ])
        poses.append(look_at(eye))
    return torch.stack(poses)


# Tanító pose-ok: az objektum 360°-os körbejárása.
# Teszt pose-ok: más eleváció, ezért novel-view kiértékelés.
train_poses = make_camera_ring(N_TRAIN, radius=2.2, elev_deg=20.0).to(device)
test_poses  = make_camera_ring(N_TEST,  radius=2.2, elev_deg=45.0,
                               azim_offset_deg=22.5).to(device)
print(f'train poses: {train_poses.shape}, test poses: {test_poses.shape}')

# Kamerabelső paraméterek.
H = W = IMG_SIZE
FOV_DEG = 40.0
FOCAL = 0.5 * W / math.tan(0.5 * math.radians(FOV_DEG))
NEAR, FAR = 1.4, 3.0
print(f'kép: {H}×{W}, focal = {FOCAL:.2f}, near/far = {NEAR}/{FAR}')


In [ ]:
# ── Sugarak előállítása egy tűlyukas kamerához (adott) ──
# Konvenciók:
#   - képkoordináták: (i, j), i = oszlop, j = sor; j = 0 a kép tetején.
#   - kamera-rendszer: -z előre, +x jobbra, +y fel.
#   - i. oszlop, j. sor pixel kamera-rendszerbeli iránya:
#         d_cam = ( (i - W/2)/focal,  -(j - H/2)/focal,  -1 )
#     (a -y előjel onnan jön, hogy a kép j-tengelye lefelé nő)
#   - a sugár origója a kamera origója a világban: c2w[:3, 3]
def get_rays(H: int, W: int, focal: float, c2w: torch.Tensor
             ) -> tuple[torch.Tensor, torch.Tensor]:
    """Tűlyukas kamera sugarainak generálása. (H, W, 3) origó és irány."""
    dev = c2w.device
    i, j = torch.meshgrid(
        torch.arange(W, dtype=torch.float32, device=dev),
        torch.arange(H, dtype=torch.float32, device=dev),
        indexing='xy')
    dirs_cam = torch.stack([
        (i - W * 0.5) / focal,
        -(j - H * 0.5) / focal,
        -torch.ones_like(i),
    ], dim=-1)                                # (H, W, 3)
    rays_d = dirs_cam @ c2w[:3, :3].T          # világ-irányok
    rays_d = rays_d / rays_d.norm(dim=-1, keepdim=True)
    rays_o = c2w[:3, 3].expand(rays_d.shape)
    return rays_o, rays_d


In [ ]:
# ── Ellenőrzés ──
_o, _d = get_rays(H, W, FOCAL, train_poses[0])
print('rays_o shape:', _o.shape, ' rays_d shape:', _d.shape)
print('középső pixel iránya (az origó felé kéne mutasson):',
      _d[H // 2, W // 2].cpu().numpy().round(3))
print('|d| ≈ 1:', _d.norm(dim=-1).mean().item())


In [ ]:
# ── 3D sugárkötegek volumetrikus renderelése (adott) ──
def render_rays(rays_o: torch.Tensor,
                rays_d: torch.Tensor,
                query_fn,
                near: float = NEAR,
                far: float = FAR,
                n_samples: int = N_SAMPLES_TRAIN,
                perturb: bool = False,
                white_bg: bool = True
                ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Köteg-sugarak vol-renderelése; query_fn(pts) → (sigma, color)."""
    prefix = rays_o.shape[:-1]
    dev = rays_o.device

    t = torch.linspace(near, far, n_samples, device=dev)
    t = t.expand(*prefix, n_samples).clone()
    if perturb:
        mids = 0.5 * (t[..., 1:] + t[..., :-1])
        upper = torch.cat([mids, t[..., -1:]], dim=-1)
        lower = torch.cat([t[..., :1], mids], dim=-1)
        t = lower + (upper - lower) * torch.rand_like(t)

    pts = rays_o[..., None, :] + rays_d[..., None, :] * t[..., None]
    sigma, color = query_fn(pts.reshape(-1, 3))
    sigma = sigma.reshape(*prefix, n_samples)
    color = color.reshape(*prefix, n_samples, 3)

    delta_segments = t[..., 1:] - t[..., :-1]
    deltas = torch.cat([delta_segments, delta_segments[..., -1:]], dim=-1)
    rgb, weights = volume_render_ray(sigma, color, deltas)

    depth   = (weights * t).sum(dim=-1)
    opacity = weights.sum(dim=-1)
    if white_bg:
        rgb = rgb + (1.0 - opacity[..., None])
    return rgb.clamp(0.0, 1.0), depth, opacity


In [ ]:
# ── Ground-truth képek előállítása az analitikus jelenetből ──
@torch.no_grad()
def render_full_image(pose: torch.Tensor, query_fn,
                      H: int = H, W: int = W, focal: float = FOCAL,
                      n_samples: int = N_SAMPLES_EVAL,
                      chunk: int = RENDER_CHUNK
                      ) -> torch.Tensor:
    """Teljes (H, W, 3) kép renderelése, csonkonkénti feldolgozással."""
    rays_o, rays_d = get_rays(H, W, focal, pose)
    rays_o = rays_o.reshape(-1, 3)
    rays_d = rays_d.reshape(-1, 3)
    rgb_chunks = []
    for k in range(0, rays_o.shape[0], chunk):
        rgb_k, _, _ = render_rays(rays_o[k:k + chunk], rays_d[k:k + chunk],
                                  query_fn, n_samples=n_samples)
        rgb_chunks.append(rgb_k)
    return torch.cat(rgb_chunks, dim=0).reshape(H, W, 3)


t0 = time.time()
gt_train = torch.stack([render_full_image(p, smiley_field) for p in train_poses])
gt_test  = torch.stack([render_full_image(p, smiley_field) for p in test_poses])
print(f'GT renderelve: {gt_train.shape[0]} train + {gt_test.shape[0]} test '
      f'({time.time() - t0:.2f}s)')

n_show = min(16, gt_train.shape[0])
cols = 4
rows = math.ceil(n_show / cols)
fig, axes = plt.subplots(rows, cols, figsize=(1.8 * cols, 1.8 * rows))
axes = np.array(axes).reshape(-1)
for ax, img in zip(axes, gt_train[:n_show].cpu()):
    ax.imshow(img.numpy())
    ax.axis('off')
for ax in axes[n_show:]:
    ax.axis('off')
plt.suptitle('Tanító GT képek (analitikus jelenet)')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, N_TEST, figsize=(2 * N_TEST, 2))
axes = np.array(axes).reshape(-1)
for ax, img in zip(axes, gt_test.cpu()):
    ax.imshow(img.numpy())
    ax.axis('off')
plt.suptitle('Teszt GT képek (más szögekből)')
plt.tight_layout(); plt.show()


In [ ]:
# ── Trilineáris interpoláció a 3D rácson (adott segédfüggvény) ──
def sample_grid_trilinear(grid: torch.Tensor,
                          points: torch.Tensor,
                          bound: float = 1.0) -> torch.Tensor:
    """Mintavételezés egy (D, H, W) vagy (D, H, W, C) rácsból trilineárisan.

    A `points` világkoordinátákban van, [-bound, +bound]^3-ban.
    Hivatalosan F.grid_sample-t hívunk; az (x, y, z) → (W, H, D) megfeleltetés
    az F.grid_sample konvenciójához igazítva.
    """
    has_C = grid.dim() == 4
    if not has_C:
        grid = grid.unsqueeze(-1)  # (D, H, W, 1)
    D, H_, W_, C = grid.shape
    grid5d = grid.permute(3, 0, 1, 2).unsqueeze(0)        # (1, C, D, H, W)
    coords = (points / bound).reshape(1, 1, 1, -1, 3)     # (x, y, z) ∈ [-1, 1]
    out = F.grid_sample(grid5d, coords, mode='bilinear',
                        padding_mode='zeros', align_corners=True)
    out = out[0, :, 0, 0, :].T                            # (N, C)
    out = out.reshape(*points.shape[:-1], C)
    if not has_C:
        out = out[..., 0]
    return out


In [ ]:
# ── Voxel-rács inicializálása ──
GRID = GRID_RES

# A nyers density-t -5.0-re inicializáljuk: softplus(-5) ≈ 0.007, így a tanítás
# közben "érintetlen" cellák gyakorlatilag zérus sűrűségen maradnak.
# Ez segít elkerülni a teljes térre kiterjedő kezdeti "ködöt".
density_grid = nn.Parameter(-5.0 * torch.ones(GRID, GRID, GRID, device=device))
color_grid   = nn.Parameter(0.5 * torch.ones(GRID, GRID, GRID, 3, device=device))
print(f'Voxel-rács: {GRID}³ cella')


In [ ]:
def voxel_query(pts: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """A tanulható voxel-rácsból mintavett (sigma, color)."""
    # TODO:
    #   1. Mintavételezd a density_grid-et a sample_grid_trilinear segítségével.
    #   2. Mintavételezd a color_grid-et is ugyanígy.
    #   3. A nyers sűrűségre alkalmazz softplus-t: sigma >= 0.
    #   4. A nyers színre alkalmazz sigmoid-ot: color ∈ [0, 1].
    raise NotImplementedError("Implementáld a voxel_query függvényt!")


In [ ]:
# ── Ellenőrzés: voxel_query alakok, tartományok, gradiens ──
pts_test = torch.rand(16, 3, device=device) * 2.0 - 1.0
sigma_test, color_test = voxel_query(pts_test)

assert sigma_test.shape == (16,), 'A sigma alakja legyen (16,)!'
assert color_test.shape == (16, 3), 'A color alakja legyen (16, 3)!'
assert torch.all(sigma_test >= 0), 'A sűrűség legyen nemnegatív!'
assert torch.all((0 <= color_test) & (color_test <= 1)), 'A szín legyen [0, 1] tartományban!'

test_loss = sigma_test.mean() + color_test.mean()
test_loss.backward()
assert density_grid.grad is not None, 'Nem folyik gradiens a density_grid felé!'
assert color_grid.grad is not None, 'Nem folyik gradiens a color_grid felé!'

density_grid.grad = None
color_grid.grad = None
print('voxel_query ellenőrzés – OK')


In [ ]:
# ── Voxel-rács tanítása ──
optim_voxel = torch.optim.Adam([density_grid, color_grid], lr=5e-2)

all_rays_o, all_rays_d, all_rgb = [], [], []
for pose, img in zip(train_poses, gt_train):
    o, d = get_rays(H, W, FOCAL, pose)
    all_rays_o.append(o.reshape(-1, 3))
    all_rays_d.append(d.reshape(-1, 3))
    all_rgb.append(img.reshape(-1, 3))
all_rays_o = torch.cat(all_rays_o, dim=0)
all_rays_d = torch.cat(all_rays_d, dim=0)
all_rgb    = torch.cat(all_rgb,    dim=0).to(device)

losses_voxel = []
pbar = tqdm(range(N_ITERS_VOXEL), desc='Voxel illesztés')
for it in pbar:
    idx = torch.randint(0, all_rays_o.shape[0], (RAY_BATCH,), device=device)
    o = all_rays_o[idx]
    d = all_rays_d[idx]
    tgt = all_rgb[idx]

    rgb, _, _ = render_rays(o, d, voxel_query,
                            n_samples=N_SAMPLES_TRAIN,
                            perturb=True)
    loss = mse_criterion(rgb, tgt)

    optim_voxel.zero_grad()
    loss.backward()
    optim_voxel.step()

    losses_voxel.append(loss.item())
    if it % 100 == 0:
        pbar.set_postfix(loss=f'{loss.item():.4f}')

plt.figure(figsize=(6, 2))
plt.plot(losses_voxel); plt.yscale('log')
plt.xlabel('iteráció'); plt.ylabel('MSE')
plt.title('Voxel-rács illesztés veszteséggörbéje')
plt.grid(True); plt.tight_layout(); plt.show()


In [ ]:
# ── Vizualizáció: voxel-rács vs GT ──
@torch.no_grad()
def render_view(pose, query_fn, n_samples=N_SAMPLES_EVAL):
    return render_full_image(pose, query_fn, n_samples=n_samples)

idxs = torch.linspace(0, N_TRAIN - 1, steps=min(4, N_TRAIN)).long().tolist()
fig, axes = plt.subplots(2, len(idxs), figsize=(2 * len(idxs), 4))
axes = np.array(axes).reshape(2, len(idxs))
for col, k in enumerate(idxs):
    axes[0, col].imshow(gt_train[k].cpu().numpy()); axes[0, col].axis('off')
    pred = render_view(train_poses[k], voxel_query).cpu().numpy()
    axes[1, col].imshow(pred); axes[1, col].axis('off')
axes[0, 0].set_title('Train GT', loc='left')
axes[1, 0].set_title('Voxel', loc='left')
plt.suptitle('Tanító nézetek')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, N_TEST, figsize=(2 * N_TEST, 4))
axes = np.array(axes).reshape(2, N_TEST)
for k in range(N_TEST):
    axes[0, k].imshow(gt_test[k].cpu().numpy()); axes[0, k].axis('off')
    pred = render_view(test_poses[k], voxel_query).cpu().numpy()
    axes[1, k].imshow(pred); axes[1, k].axis('off')
axes[0, 0].set_title('Test GT', loc='left')
axes[1, 0].set_title('Voxel', loc='left')
plt.suptitle('Novel-view szintézis (új szögekből)')
plt.tight_layout(); plt.show()


---

## 3. Rész — Tiny-NeRF

A voxel-rács sok memóriát fogyaszt, és a felbontását korán fixálni kell. Az
**implicit reprezentáció** ötletet egy kis MLP-vel valósítjuk meg: a hálózat
egyetlen 3D pontot kap és egy $(\sigma, \mathbf{c})$ párt ad vissza.

Mivel az MLP-k önmagukban hajlamosak túl sima függvényeket tanulni, a finom
részleteket **pozicionális kódolással** segítjük: a koordinátákat több
frekvenciájú $\sin/\cos$ függvényekkel bővítjük (NeRF-stílusban).

A hálózat ugyanazt a `query_fn` interfészt fogja használni, mint a voxel-rács:

```python
query_fn(pts) -> sigma, color
```

Ezért a `render_rays` függvényt változtatás nélkül újra tudjuk használni.


In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (NeRF-stílus)."""

    def __init__(self, num_freqs: int = 6):
        super().__init__()
        self.L = num_freqs
        freqs = (2.0 ** torch.arange(num_freqs, dtype=torch.float32)) * math.pi
        self.register_buffer('freqs', freqs)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (..., 3) → (..., 3 + 6L) kódolt jellemzők."""
        # TODO:
        #   scaled: (..., 3, L)
        #   sin:    (..., 3*L)
        #   cos:    (..., 3*L)
        #   out:    (..., 3 + 6*L)
        # Tipp: scaled = x[..., None] * self.freqs
        raise NotImplementedError("Implementáld a PositionalEncoding.forward-ot!")

    @property
    def out_dim(self) -> int:
        return 3 + 3 * 2 * self.L


In [ ]:
# ── Ellenőrzés ──
_pe = PositionalEncoding(num_freqs=6).to(device)
_x = torch.randn(5, 3, device=device)
_y = _pe(_x)
print(f'PE bemenet: {_x.shape}, kimenet: {_y.shape}, várt: (5, {_pe.out_dim})')
assert _y.shape == (5, _pe.out_dim), 'HIBA a PositionalEncoding kimenetén!'
print('PE ellenőrzés – OK')


In [ ]:
class TinyNeRF(nn.Module):
    """Egy kicsi MLP, ami (x, y, z) → (sigma, rgb)-t ad."""

    def __init__(self, num_freqs: int = 6,
                 hidden: int = 128,
                 n_layers: int = 4):
        super().__init__()
        self.pe = PositionalEncoding(num_freqs)
        in_dim = self.pe.out_dim
        layers = [nn.Linear(in_dim, hidden), nn.ReLU(inplace=True)]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.ReLU(inplace=True)]
        self.trunk = nn.Sequential(*layers)
        self.sigma_head = nn.Linear(hidden, 1)
        nn.init.constant_(self.sigma_head.bias, -2.0)
        self.color_head = nn.Linear(hidden, 3)

    def forward(self, pts: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """pts: (..., 3) → (sigma (...,), color (..., 3))."""
        # TODO:
        #   1. encoded = self.pe(pts)
        #   2. h = self.trunk(encoded)
        #   3. sigma_raw = self.sigma_head(h)
        #   4. color_raw = self.color_head(h)
        #   5. sigma = softplus(sigma_raw), az utolsó dimenzió lenyomásával
        #   6. color = sigmoid(color_raw)
        raise NotImplementedError("Implementáld a TinyNeRF.forward-ot!")


In [ ]:
# ── Ellenőrzés: TinyNeRF kimenetek ──
_test_model = TinyNeRF(num_freqs=4, hidden=32, n_layers=2).to(device)
_pts = torch.randn(7, 3, device=device)
_sigma, _color = _test_model(_pts)

assert _sigma.shape == (7,), 'A sigma alakja legyen (7,)!'
assert _color.shape == (7, 3), 'A color alakja legyen (7, 3)!'
assert torch.all(_sigma >= 0), 'A sigma legyen nemnegatív!'
assert torch.all((0 <= _color) & (_color <= 1)), 'A color legyen [0, 1] tartományban!'

_test_loss = _sigma.mean() + _color.mean()
_test_loss.backward()
assert any(p.grad is not None for p in _test_model.parameters()),     'Nem folyik gradiens a TinyNeRF paraméterei felé!'
print('TinyNeRF ellenőrzés – OK')


In [ ]:
# ── TinyNeRF tanítása ──
model_nerf = TinyNeRF(num_freqs=6, hidden=NERF_HIDDEN, n_layers=NERF_LAYERS).to(device)
print(f'TinyNeRF méret: hidden={NERF_HIDDEN}, rétegek={NERF_LAYERS}')
print(f'Paraméterek: {sum(p.numel() for p in model_nerf.parameters()):,}')

optim_nerf = torch.optim.Adam(model_nerf.parameters(), lr=5e-4)


def mlp_query(pts: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    return model_nerf(pts)


model_nerf.train()
losses_nerf = []
pbar = tqdm(range(N_ITERS_NERF), desc='TinyNeRF tanítás')
for it in pbar:
    idx = torch.randint(0, all_rays_o.shape[0], (RAY_BATCH,), device=device)
    o = all_rays_o[idx]
    d = all_rays_d[idx]
    tgt = all_rgb[idx]

    rgb, _, _ = render_rays(o, d, mlp_query,
                            n_samples=N_SAMPLES_TRAIN,
                            perturb=True)
    loss = mse_criterion(rgb, tgt)

    optim_nerf.zero_grad()
    loss.backward()
    optim_nerf.step()

    losses_nerf.append(loss.item())
    if it % 200 == 0:
        pbar.set_postfix(loss=f'{loss.item():.4f}')

plt.figure(figsize=(6, 2))
plt.plot(losses_nerf); plt.yscale('log')
plt.xlabel('iteráció'); plt.ylabel('MSE')
plt.title('TinyNeRF tanítási veszteség')
plt.grid(True); plt.tight_layout(); plt.show()


In [ ]:
# ── Novel-view kiértékelés (a tesztpózok TANÍTÁS KÖZBEN NEM láttak) ──
model_nerf.eval()

fig, axes = plt.subplots(2, N_TEST, figsize=(2 * N_TEST, 4))
axes = np.array(axes).reshape(2, N_TEST)
for k in range(N_TEST):
    axes[0, k].imshow(gt_test[k].cpu().numpy()); axes[0, k].axis('off')
    pred = render_view(test_poses[k], mlp_query, n_samples=N_SAMPLES_EVAL).cpu().numpy()
    axes[1, k].imshow(pred); axes[1, k].axis('off')
axes[0, 0].set_title('Test GT', loc='left')
axes[1, 0].set_title('TinyNeRF', loc='left')
plt.suptitle('Novel-view szintézis (új szögekből)')
plt.tight_layout(); plt.show()


def psnr(a, b):
    """Egyszerű PSNR-mutató két [0,1] tartományú kép között."""
    return -10.0 * torch.log10(mse_criterion(a, b))


with torch.no_grad():
    test_psnrs = []
    for pose, gt in zip(test_poses, gt_test):
        pred = render_view(pose, mlp_query, n_samples=N_SAMPLES_EVAL)
        test_psnrs.append(psnr(pred, gt.to(device)).item())
print(f'Teszt PSNR átlag: {np.mean(test_psnrs):.2f} dB '
      f'(min {np.min(test_psnrs):.2f}, max {np.max(test_psnrs):.2f})')


---

## 4. Rész (opcionális) — Gauss-primitívmező, 3DGS-inspirált

A 2-3. részben a smiley sűrűségét és színét egy voxel-rács vagy egy MLP
definiálta minden 3D pontra. Egy másik reprezentációs családot az
**explicit primitív-alapú** módszerek alkotnak: itt a modell konkrét
primitívek listája, és a tér bármely pontjának értékét a primitívek
hozzájárulásainak összege adja.

A 18. előadáson tárgyalt **3D Gaussian Splatting (3DGS)** is ebbe az irányba
mutat. Itt viszont nem teljes 3DGS-rasterizálót írunk, hanem egy sokkal
egyszerűbb, volumetrikusan renderelt Gauss-primitívmezőt.

> Fontos: ez **nem** teljes 3DGS. A valódi 3DGS képernyőtérben rasterizál,
> anizotróp Gaussokat és gyakran gömbi harmonikus színeket használ. Ez a rész
> csak az explicit Gauss-primitíves reprezentáció gondolatát mutatja meg egy
> már meglévő volumetrikus rendererrel.


In [ ]:
class GaussianSplat(nn.Module):
    """Tanulható, izotróp 3D Gauss-primitívekből álló mező."""
    DENSITY_SCALE = 30.0

    def __init__(self, n: int = 256):
        super().__init__()
        self.positions = nn.Parameter(0.6 * (torch.rand(n, 3) - 0.5))
        self.colors_raw = nn.Parameter(torch.randn(n, 3) * 0.3)
        self.opacities_raw = nn.Parameter(-3.0 * torch.ones(n))
        self.log_radii = nn.Parameter(math.log(0.05) * torch.ones(n))

    def forward(self, pts: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """pts: (P, 3) → (sigma (P,), color (P, 3))."""
        # Páronkénti négyzetes távolság: d² = ‖p‖² - 2 p·c + ‖c‖².
        # Így nincs (P, N, 3) tenzor, de a (P, N) mátrix még így is nagy lehet.
        d2 = (
            (pts ** 2).sum(-1, keepdim=True)
            - 2.0 * (pts @ self.positions.T)
            + (self.positions ** 2).sum(-1)
        )

        radii_sq    = torch.exp(2.0 * self.log_radii)
        contrib     = torch.exp(-d2 / (2.0 * radii_sq + 1e-8))
        opacities   = F.softplus(self.opacities_raw)
        density_per = self.DENSITY_SCALE * opacities * contrib

        sigma = density_per.sum(dim=-1)
        colors = torch.sigmoid(self.colors_raw)
        rgb_num = density_per @ colors
        color = rgb_num / (sigma[..., None] + 1e-8)
        return sigma, color


In [ ]:
# ── Splat tanítása (opcionális) ──
splat = GaussianSplat(n=N_GAUSS_SPLAT).to(device)
print(f'Gauss-primitívek száma: {N_GAUSS_SPLAT}')

optim_splat = torch.optim.Adam([
    {'params': [splat.positions],                                        'lr': 1e-2},
    {'params': [splat.colors_raw, splat.opacities_raw, splat.log_radii], 'lr': 5e-2},
])


def splat_query(pts):
    return splat(pts)


losses_splat = []
pbar = tqdm(range(N_ITERS_SPLAT), desc='Splat illesztés')
for it in pbar:
    idx = torch.randint(0, all_rays_o.shape[0], (RAY_BATCH_SPLAT,), device=device)
    o = all_rays_o[idx]
    d = all_rays_d[idx]
    tgt = all_rgb[idx]

    rgb, _, _ = render_rays(o, d, splat_query,
                            n_samples=N_SAMPLES_SPLAT,
                            perturb=True)
    loss = mse_criterion(rgb, tgt)

    optim_splat.zero_grad()
    loss.backward()
    optim_splat.step()

    losses_splat.append(loss.item())
    if it % 100 == 0:
        pbar.set_postfix(loss=f'{loss.item():.4f}')

plt.figure(figsize=(6, 2))
plt.plot(losses_splat); plt.yscale('log')
plt.xlabel('iteráció'); plt.ylabel('MSE')
plt.title('Gauss-primitívmező tanítási görbe'); plt.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
# ── Vizualizáció: Gauss-primitívmező vs GT (opcionális) ──
@torch.no_grad()
def render_splat_view(pose):
    return render_full_image(pose, splat_query,
                             n_samples=N_SAMPLES_SPLAT_EVAL,
                             chunk=RENDER_CHUNK_SPLAT).cpu().numpy()

idxs = torch.linspace(0, N_TRAIN - 1, steps=min(4, N_TRAIN)).long().tolist()
fig, axes = plt.subplots(2, len(idxs), figsize=(2 * len(idxs), 4))
axes = np.array(axes).reshape(2, len(idxs))
for col, k in enumerate(idxs):
    axes[0, col].imshow(gt_train[k].cpu().numpy()); axes[0, col].axis('off')
    axes[1, col].imshow(render_splat_view(train_poses[k])); axes[1, col].axis('off')
axes[0, 0].set_title('Train GT', loc='left')
axes[1, 0].set_title('Gauss', loc='left')
plt.suptitle('Tanító nézetek')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, N_TEST, figsize=(2 * N_TEST, 4))
axes = np.array(axes).reshape(2, N_TEST)
for k in range(N_TEST):
    axes[0, k].imshow(gt_test[k].cpu().numpy()); axes[0, k].axis('off')
    axes[1, k].imshow(render_splat_view(test_poses[k])); axes[1, k].axis('off')
axes[0, 0].set_title('Test GT', loc='left')
axes[1, 0].set_title('Gauss', loc='left')
plt.suptitle('Novel-view szintézis (új szögekből)')
plt.tight_layout(); plt.show()


---

## Beadás

A kitöltött `.ipynb` fájlt töltsd fel a Moodle-be a megadott határidőig
(2026.05.27.). Ügyelj rá, hogy a notebook **lefuttatva** kerüljön beadásra
(a kimenetek látsszanak), és minden kötelező TODO blokk implementálva legyen.

A beadáshoz a `FAST_MODE=False` beállítás javasolt, a `FAST_MODE=True`
debugginghoz javasolt, illetve ha lokálban futattunk és nem rendelkezünkkellően erős GPU-val.

## Kötelező ellenőrzőlista

- [ ] `compute_alphas` működik, és az ellenőrző cella lefut.
- [ ] `compute_transmittance` működik, és az ellenőrző cella lefut.
- [ ] `volume_render_ray` működik, és az ellenőrző cella lefut.
- [ ] A voxel-rács tanítása lefut, és látszik a veszteséggörbe.
- [ ] A voxel-rácsos train/test vizualizációk megjelennek.
- [ ] `PositionalEncoding.forward` működik.
- [ ] `TinyNeRF.forward` működik.
- [ ] A TinyNeRF tanítása lefut, és látszik legalább néhány tesztnézet.

## Rövid megválaszolandó kérdések

A notebook végére, egy külön markdown cellába válaszolj röviden:

**Hogyan értékelnéd a voxeles és neurális reprzenetációval kapott eredményeket? Melyik reprezentáció teljesít jobban és szerinted miért?**

Az opcionális 4. rész nem kötelező. Ha lefuttatod, vesd össze röviden a
voxel-rács, a TinyNeRF és a Gauss-primitívmező erősségeit és gyengéit is.
